<a href="https://colab.research.google.com/github/LeninGF/IAG-2024B-GenerativeQA/blob/main/question-answering-Bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question Answering Generative

- Coder: Lenin G. Falconí
- Date: 2025-01-27

https://huggingface.co/docs/transformers/en/tasks/question_answering

## Instalación de Librerías

In [1]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

In [2]:
# For Colab
!pip install python-dotenv
!pip install huggingface_hub
!pip install datasets

## Login en Huggingfaces

Se realiza el login usando archivo .env

In [3]:
# For Colab
from dotenv import load_dotenv
import os
dotenv_path = '/content/.env'
load_dotenv(dotenv_path)

True

In [4]:
import os
from huggingface_hub import login
token = os.getenv('HUGGINGFACE_TOKEN')
login(token)

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## Carga del Dataset

Se procede a realizar la carga del dataset desde huggingface

In [12]:
from datasets import load_dataset
path2dataset = "LeninGF/robos-question-answering"
squad = load_dataset(path2dataset)

In [13]:
squad

DatasetDict({
    train: Dataset({
        features: ['index', 'context', 'question', 'answer_start', 'answer_end', 'impossible_find_answer', 'answer_text', 'is_impossible', 'context_id', 'answer_text_number_words'],
        num_rows: 4572
    })
})

In [14]:
# Split the dataset (adjust test_size=0.2 as needed)
squad = squad["train"].train_test_split(test_size=0.2, seed=42)

In [15]:
squad

DatasetDict({
    train: Dataset({
        features: ['index', 'context', 'question', 'answer_start', 'answer_end', 'impossible_find_answer', 'answer_text', 'is_impossible', 'context_id', 'answer_text_number_words'],
        num_rows: 3657
    })
    test: Dataset({
        features: ['index', 'context', 'question', 'answer_start', 'answer_end', 'impossible_find_answer', 'answer_text', 'is_impossible', 'context_id', 'answer_text_number_words'],
        num_rows: 915
    })
})

In [16]:
squad["train"][0]

{'index': 466,
 'context': 'señor fiscal el dia ayer 20 de junio del 2018 a las 17h00 aproximadamente por el redondel de jaramijo donde esta la fabrica puerto mar del canton jaramijo yo iba caminando y hablando por telefono de repetente un sujeto que iva caminando me arrancho el telefono de maneta violenta y lego salio corriendo con mi telefono celular era una persona joven alta moreno y de contextura delgada solocito se pida al ecu 911 si existen camara en el lugar a fin de identificar a la persona que me robo anexo a mi denuncia factura del telefono donde consta sus caracteristicas',
 'question': '¿En qué fecha ocurrió el incidente?',
 'answer_start': 25,
 'answer_end': 45,
 'impossible_find_answer': False,
 'answer_text': '20 de junio del 2018',
 'is_impossible': '0',
 'context_id': 'context_93',
 'answer_text_number_words': 5}

In [17]:
from transformers import AutoTokenizer
# model = "distilbert/distilbert-base-uncased"
model = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

## Cargando Modelo Pre-Entrenado

Se considera utilizar el modelo `dccuchile/bert-base-spanish-wwm-cased` que sería un ajuste al Español

In [18]:
from transformers import AutoModelForQuestionAnswering
model = AutoModelForQuestionAnswering.from_pretrained(model)

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Preprocesamiento

In [19]:
def preprocess_function(examples):
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        truncation=True,
        padding="max_length",
        max_length=384,  # Adjust if needed
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer_start = examples["answer_start"][sample_idx]
        answer_end = examples["answer_end"][sample_idx]
        answer_text = examples["answer_text"][sample_idx]

        # Handle impossible answers (if your dataset has them)
        if examples["is_impossible"][sample_idx] or answer_start == -1:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Logic to find token positions (same as before)
            sequence_ids = inputs.sequence_ids(i)
            context_start = 0
            while sequence_ids[context_start] != 1:
                context_start += 1
            context_end = len(sequence_ids) - 1
            while sequence_ids[context_end] != 1:
                context_end -= 1

            if (answer_start < offsets[context_start][0] or
                answer_end > offsets[context_end][1]):
                start_positions.append(0)
                end_positions.append(0)
            else:
                # Find token positions for answer
                start_token = context_start
                while start_token < context_end and offsets[start_token][0] <= answer_start:
                    start_token += 1
                end_token = context_end
                while end_token >= context_start and offsets[end_token][1] >= answer_end:
                    end_token -= 1
                start_positions.append(start_token - 1)
                end_positions.append(end_token + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Apply preprocessing
tokenized_dataset = squad.map(
    preprocess_function,
    batched=True,
    remove_columns=squad["train"].column_names  # Remove unused columns
)

Map:   0%|          | 0/3657 [00:00<?, ? examples/s]

Map:   0%|          | 0/915 [00:00<?, ? examples/s]

In [20]:
# from transformers import DefaultDataCollator

# data_collator = DefaultDataCollator()

In [21]:
# from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

# model = AutoModelForQuestionAnswering.from_pretrained("distilbert/distilbert-base-uncased")

## Evaluación del Rendimiento del Modelo
Se utilizara `squad_metric` para calcular EM y F1

In [26]:
!pip install evaluate # for colab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.7 MB/s eta 0:00:00


In [27]:
import evaluate
from tqdm.auto import tqdm

qa_metric = evaluate.load("squad")

def compute_metrics(start_logits, end_logits, tokenized_dataset, original_dataset):
    # Map predicted token positions to text spans
    predicted_answers = []
    for i in tqdm(range(len(tokenized_dataset))):
        start_logit = start_logits[i]
        end_logit = end_logits[i]
        offsets = tokenized_dataset[i]["offset_mapping"]
        example_id = tokenized_dataset[i]["example_ids"]

        # Get the original example
        original_example = original_dataset[example_id]

        # Find the predicted token positions
        start_index = np.argmax(start_logit)
        end_index = np.argmax(end_logit)

        # Convert tokens to character positions
        start_char = offsets[start_index][0]
        end_char = offsets[end_index][1]

        # Extract predicted answer text
        predicted_text = original_example["context"][start_char:end_char]

        # Handle impossible answers (if applicable) Actualizar a la etiqueta correcta
        if original_example["is_impossible"]:
            predicted_text = ""

        predicted_answers.append({
            "id": str(example_id),
            "prediction_text": predicted_text,
            "no_answer_probability": 0.0 if predicted_text else 1.0,
        })

    # Format ground truth answers
    true_answers = [
        {"id": str(i), "answers": {"text": [ex["answer_text"]], "answer_start": [ex["answer_start"]]}}
        for i, ex in enumerate(original_dataset)
    ]

    # Compute metrics
    results = qa_metric.compute(predictions=predicted_answers, references=true_answers)
    return results

Para evaluar durante el entrenamiento se defina la clase `QATrainer`

In [28]:
from transformers import Trainer

class QATrainer(Trainer):
    def compute_metrics(self, eval_preds):
        start_logits, end_logits = eval_preds.predictions
        return compute_metrics(start_logits, end_logits, self.eval_dataset, dataset["test"])

## Entrenamiento

In [22]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [36]:
os.mkdir("./models")

In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./fge-robos-qa-model", # colocar dentro de models
    evaluation_strategy="epoch",
    learning_rate=3e-5,  # Slightly lower for non-English models
    per_device_train_batch_size=8,  # Adjust based on GPU memory
    per_device_eval_batch_size=8,
    num_train_epochs=10,  # Spanish datasets may need more epochs
    weight_decay=0.01,
    save_strategy="epoch",
    fp16=True,  # Use if GPU supports it
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [29]:
from transformers import Trainer

trainer = QATrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],  # Now using the test split
    tokenizer=tokenizer,
)

<ipython-input-29-350cb3dab31a>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `QATrainer.__init__`. Use `processing_class` instead.
  trainer = QATrainer(


In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.319714
2,0.494900,0.283443
3,0.313900,0.317269
4,0.277100,0.366507
5,0.190000,0.486700
6,0.151400,0.491661
7,0.107200,0.522017
8,0.069900,0.580718
9,0.047700,0.648142
10,0.021700,0.659826


TrainOutput(global_step=4580, training_loss=0.18319864627055205, metrics={'train_runtime': 1021.8231, 'train_samples_per_second': 35.789, 'train_steps_per_second': 4.482, 'total_flos': 7166716795376640.0, 'train_loss': 0.18319864627055205, 'epoch': 10.0})

## Evaluación


In [31]:
# Evaluate on test set
results = trainer.evaluate()
print("Test set results:", results)

Test set results: {'eval_loss': 0.6598264575004578, 'eval_runtime': 7.3955, 'eval_samples_per_second': 123.724, 'eval_steps_per_second': 15.55, 'epoch': 10.0}


## Guardando Modelo y Tokenizer

In [33]:
os.mkdir("fge-qa-model")
model.save_pretrained(".fge-qa-model/fine-tuned-qa-model")
tokenizer.save_pretrained(".fge-qa-model/fine-tuned-qa-model")

('.fge-qa-model/fine-tuned-qa-model/tokenizer_config.json',
 '.fge-qa-model/fine-tuned-qa-model/special_tokens_map.json',
 '.fge-qa-model/fine-tuned-qa-model/vocab.txt',
 '.fge-qa-model/fine-tuned-qa-model/added_tokens.json',
 '.fge-qa-model/fine-tuned-qa-model/tokenizer.json')

In [37]:
trainer.push_to_hub()

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

events.out.tfevents.1738602795.06c05ddb276d.262.0:   0%|          | 0.00/10.1k [00:00<?, ?B/s]

Upload 4 LFS files:   0%|          | 0/4 [00:00<?, ?it/s]

events.out.tfevents.1738603923.06c05ddb276d.262.1:   0%|          | 0.00/359 [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.30k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/LeninGF/fge-robos-qa-model/commit/e47b7e9c2af81dedafa2e088330816f4f7fcfe1a', commit_message='End of training', commit_description='', oid='e47b7e9c2af81dedafa2e088330816f4f7fcfe1a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/LeninGF/fge-robos-qa-model', endpoint='https://huggingface.co', repo_type='model', repo_id='LeninGF/fge-robos-qa-model'), pr_revision=None, pr_num=None)

## Demo

In [38]:
question = "¿En qué fecha ocurrió el incidente?"
context = "es el caso señor fiscal que el dia de hoy 28 de julio del 2016 siendo aproximadamente las 17h00 en circunstancias que me baje de un bus en la parroquia san camilo con la finalidad de dirigirme a mi lugar de trabajo esto el taller eco frio de repente al llegar a la altura del cuerpo de bomberos fui inteceptado por dos sujetos inidentificados que se movilizaban a bordo de una motocicleta marca suzuki colo rojo sin placas los mismos que con un arma de fuego me intimidaron acto seguido procedieron a robarme ciento ochenta dolares en efectivo dinero que era de producto de mi trabajo luego se dieron a la fuga con rumbo desconocido por tal motivo solicito se realicen las respectivas investigaciones"

In [39]:
from transformers import pipeline

question_answerer = pipeline("question-answering", model="LeninGF/fge-robos-qa-model")
question_answerer(question=question, context=context)

config.json:   0%|          | 0.00/716 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/730k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Device set to use cuda:0


{'score': 1.8710013094391797e-08,
 'start': 28,
 'end': 62,
 'answer': 'el dia de hoy 28 de julio del 2016'}